# Document Question Answering System (RAG)
### Retrieval-Augmented Generation over Custom Documents

**Author:** Aashika Pandey
**Project:** Week 7 — Document QA System (RAG)

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline: instead
of letting a language model answer purely from what it memorized during
training, the system pulls relevant passages from a document and answers using
that context. That's what makes it usable for private or domain-specific data
the model was never trained on.

The pipeline follows the standard RAG architecture:

1. Document Ingestion
2. Text Chunking
3. Embedding Creation
4. Vector Database (storage + similarity search)
5. Query Processing
6. Context Retrieval
7. Answer Generation

I built each stage as its own module, mostly so I could debug them one at a
time and swap pieces around later (a different embedding model, a different
chunk size, hybrid retrieval) without rewriting everything else.

## 1. Environment Setup

Libraries used:
- `huggingface_hub` to download the dataset files directly from the Hub
- `sentence-transformers` for the embedding model
- `faiss-cpu` as the vector database
- `transformers` for the answer-generation model
- `rank-bm25` for keyword-based retrieval, used later in the hybrid search
- `numpy`, `textwrap` for general utilities

**Dataset:** [`vectara/open_ragbench`](https://huggingface.co/datasets/vectara/open_ragbench),
the *Open RAG Benchmark*, built from roughly 1000 arXiv PDF papers. Each paper
arrives pre-split into sections (text, tables, images) and comes with
human/LLM-generated queries, gold relevance labels (`qrels.json`, which
section actually answers each query), and reference answers (`answers.json`).
I picked this dataset specifically because those gold labels let me measure
retrieval accuracy for real in Section 10, instead of just reading the outputs
and guessing whether they look right. License is CC-BY-NC-4.0, non-commercial
use, which is fine for coursework.

In [ ]:
!pip install -q huggingface_hub sentence-transformers faiss-cpu transformers rank-bm25 numpy

In [ ]:
import os
import re
import json
import random
import textwrap
import numpy as np

from huggingface_hub import hf_hub_download, list_repo_files
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi
from transformers import pipeline

import warnings
warnings.filterwarnings("ignore")  # transformers is noisy about model warnings, muting it

random.seed(42)  # keeping this fixed so the sampled papers/queries stay the same on reruns
print("all set, imports done")

## 2. Document Ingestion Module

This module downloads and parses documents straight from the
`vectara/open_ragbench` dataset on the Hub, so there's no manual PDF upload
step. The dataset already ships papers pre-parsed into structured JSON, which
means the messy OCR/text-extraction part of ingestion has effectively been
done already.

The full corpus is about 1000 papers (743 MB), which is a lot more than a
beginner project needs to prove the pipeline works, so I'm pulling down a
smaller sample instead. That keeps the notebook runnable in a few minutes
while still giving a real evaluation to work with.

In [ ]:
REPO_ID = "vectara/open_ragbench"
REPO_TYPE = "dataset"
BASE_PATH = "official/pdf/arxiv"


def hf_file(relative_path):
    # thin wrapper so I'm not repeating repo_id/repo_type everywhere below
    return hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=relative_path)


# these three are small, so pulling them in full is fine
queries_path = hf_file(BASE_PATH + "/queries.json")
qrels_path = hf_file(BASE_PATH + "/qrels.json")
answers_path = hf_file(BASE_PATH + "/answers.json")

with open(queries_path) as f:
    all_queries = json.load(f)
with open(qrels_path) as f:
    all_qrels = json.load(f)
with open(answers_path) as f:
    all_answers = json.load(f)

print("queries:", len(all_queries))
print("qrels (query -> gold section):", len(all_qrels))
print("reference answers:", len(all_answers))

Rather than downloading all ~1000 corpus JSON files, I selected a sample of
`N_PAPERS` gold ("positive") documents, meaning papers that actually serve as
the correct answer source for at least one query, and downloaded only those.
That keeps the notebook runnable in a few minutes while still testing
retrieval against real gold labels.

In [ ]:
N_PAPERS = 15  # bump this up if you want a more thorough eval, just slower to download

# only the papers that are actually a gold answer for at least one query
gold_doc_ids = list({v["doc_id"] for v in all_qrels.values()})
sample_doc_ids = random.sample(gold_doc_ids, min(N_PAPERS, len(gold_doc_ids)))

print("sampling", len(sample_doc_ids), "out of", len(gold_doc_ids), "gold documents")

corpus_paths = {}
for doc_id in sample_doc_ids:
    corpus_paths[doc_id] = hf_file(BASE_PATH + "/corpus/" + doc_id + ".json")

print("downloaded", len(corpus_paths), "paper JSON files")

In [ ]:
def flatten_section(section):
    # tables come as separate markdown blobs in the raw JSON, folding them into
    # the section text so they don't get lost during chunking. skipping images
    # for now -- see the multimodal note near the end
    parts = [section.get("text", "")]
    for table_id, table_md in (section.get("tables") or {}).items():
        parts.append("\n[Table " + str(table_id) + "]\n" + table_md)
    return "\n".join(p for p in parts if p)


documents = {}     # doc_id -> list of section texts
doc_titles = {}     # doc_id -> title, just for nicer print statements later

for doc_id, path in corpus_paths.items():
    with open(path) as f:
        paper = json.load(f)
    doc_titles[doc_id] = paper.get("title", doc_id)
    documents[doc_id] = [flatten_section(s) for s in paper.get("sections", [])]

for doc_id in list(documents)[:3]:
    print(doc_id, "--", doc_titles[doc_id], "--", len(documents[doc_id]), "sections")

## 3. Text Chunking

Long documents get split into smaller, overlapping chunks before embedding.
The overlap matters: without it, a sentence sitting right at a chunk boundary
gets cut in half, and neither half retrieves well on its own. With overlap,
that sentence still shows up whole in the neighboring chunk.

`chunk_size` and `chunk_overlap` are the two knobs worth tuning. Smaller
chunks retrieve more precisely but lose surrounding context; larger chunks
keep more context but dilute relevance.

In [ ]:
def chunk_text(text, chunk_size=700, chunk_overlap=100):
    # collapse whitespace/newlines first, messy PDF text otherwise throws off
    # the character counting below
    text = re.sub(r"\s+", " ", text).strip()

    chunks = []
    pos = 0
    step = chunk_size - chunk_overlap
    while pos < len(text):
        piece = text[pos:pos + chunk_size]
        if piece.strip():
            chunks.append(piece)
        pos += step
    return chunks


all_chunks = []
chunk_sources = []  # {doc_id, section_id} per chunk -- needed later to check
                    # retrieved chunks against the dataset's gold qrels

for doc_id, sections in documents.items():
    for section_id, section_text in enumerate(sections):
        for piece in chunk_text(section_text, chunk_size=700, chunk_overlap=100):
            all_chunks.append(piece)
            chunk_sources.append({"doc_id": doc_id, "section_id": section_id})

print("total chunks:", len(all_chunks))
print("\nsample chunk:\n")
print(textwrap.fill(all_chunks[0], width=90))

## 4. Embedding Creation

Each chunk gets converted into a dense vector using a pre-trained sentence
embedding model. I went with `all-MiniLM-L6-v2` here: it's small, fast, produces
384-dimensional vectors, and is a common default for RAG prototypes, which
makes it a reasonable fit for a project this size without needing a GPU.

In [ ]:
embedding_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embedding_model_name)

# this is the slow-ish step -- encoding every chunk once, upfront
chunk_embeddings = embedder.encode(all_chunks, show_progress_bar=True, convert_to_numpy=True)
embedding_dim = chunk_embeddings.shape[1]

print("model:", embedding_model_name)
print("chunk embeddings:", chunk_embeddings.shape[0])
print("embedding dim:", embedding_dim)

## 5. Vector Database

The chunk embeddings get stored in a FAISS index for fast similarity search.
I used `IndexFlatL2` here, an exact nearest-neighbor search that's simple and
reliable at small-to-medium scale. For much larger corpora, an approximate
index like `IndexIVFFlat` or `IndexHNSWFlat` would trade a bit of accuracy for
speed, but that's overkill at this scale.

In [ ]:
index = faiss.IndexFlatL2(embedding_dim)
index.add(chunk_embeddings)

print("FAISS index ready --", index.ntotal, "vectors stored")

## 6. Query Processing

The user's question gets converted into an embedding using the same model
used for the document chunks. Same model means the query and the chunks land
in the same vector space, so comparing them actually means something.

In [ ]:
def embed_query(query):
    # single-item list because .encode expects a batch, even for one string
    return embedder.encode([query], convert_to_numpy=True)

## 7. Context Retrieval Module

Given a query embedding, this pulls the top-k most relevant chunks from the
FAISS index (pure vector search). I also added a BM25 keyword retriever
alongside it, and a hybrid function that blends both. That's the "hybrid
search (keyword + vector)" optimization the brief calls out.

In [ ]:
def vector_search(query, top_k=3):
    # plain FAISS nearest-neighbor lookup, no keyword matching involved here
    q_vec = embed_query(query)
    distances, indices = index.search(q_vec, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "chunk": all_chunks[idx],
            "doc_id": chunk_sources[idx]["doc_id"],
            "section_id": chunk_sources[idx]["section_id"],
            "distance": float(distances[0][rank]),
        })
    return results


# BM25 needs the corpus tokenized up front -- doing that once here rather
# than inside keyword_search so it's not repeated on every call
tokenized_chunks = [c.lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)


def keyword_search(query, top_k=3):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{
        "chunk": all_chunks[i],
        "doc_id": chunk_sources[i]["doc_id"],
        "section_id": chunk_sources[i]["section_id"],
        "score": float(scores[i]),
    } for i in top_idx]


def _minmax_normalize(score_dict):
    # squashes scores to [0, 1] so vector distances and BM25 scores
    # (totally different scales) can actually be added together
    vals = np.array(list(score_dict.values()))
    if vals.max() == vals.min():
        return {k: 0.0 for k in score_dict}
    return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in score_dict.items()}


def hybrid_search(query, top_k=3, alpha=0.5):
    """Blend vector similarity with BM25 keyword scores. alpha=1 is pure
    vector search, alpha=0 is pure BM25, 0.5 splits it evenly."""
    q_vec = embed_query(query)
    distances, indices = index.search(q_vec, len(all_chunks))
    vec_scores = {idx: 1 / (1 + dist) for idx, dist in zip(indices[0], distances[0])}

    bm25_scores_all = bm25.get_scores(query.lower().split())
    bm25_scores = {i: s for i, s in enumerate(bm25_scores_all)}

    vec_norm = _minmax_normalize(vec_scores)
    bm25_norm = _minmax_normalize(bm25_scores)

    combined = {}
    for i in range(len(all_chunks)):
        combined[i] = alpha * vec_norm.get(i, 0) + (1 - alpha) * bm25_norm.get(i, 0)

    top_idx = sorted(combined, key=combined.get, reverse=True)[:top_k]
    return [{
        "chunk": all_chunks[i],
        "doc_id": chunk_sources[i]["doc_id"],
        "section_id": chunk_sources[i]["section_id"],
        "score": combined[i],
    } for i in top_idx]

## 8. Answer Generation

The retrieved chunks get joined into one context block and passed to a
language model alongside the original question, so the final answer is
grounded in the retrieved text rather than whatever the model happens to
remember.

I used `google/flan-t5-base` here since it's instruction-tuned, runs fine on
CPU, and is a common lightweight choice for RAG prototypes. Swap in a bigger
model, or an API-based one, if you have GPU or API access.

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")


def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(c["chunk"] for c in retrieved_chunks)
    return (
        "Answer the question using only the context below. "
        "If the answer isn't in the context, say you don't know.\n\n"
        "Context:\n" + context + "\n\n"
        "Question: " + query + "\n"
        "Answer:"
    )


def answer_question(query, top_k=3, mode="hybrid"):
    # mode picks which retriever feeds the context -- "hybrid" is the default
    # since that's what performed best when I compared them
    if mode == "vector":
        retrieved = vector_search(query, top_k)
    elif mode == "keyword":
        retrieved = keyword_search(query, top_k)
    else:
        retrieved = hybrid_search(query, top_k)

    prompt = build_prompt(query, retrieved)
    generated = generator(prompt, max_new_tokens=150)[0]["generated_text"]

    return {
        "query": query,
        "answer": generated,
        "retrieved_chunks": retrieved,
        "mode": mode,
    }

## 9. Example Run

Here's a worked example using a real question from the dataset, one whose
gold answer document happens to be in the sample I downloaded.

In [ ]:
# picking the first query whose gold document actually landed in our sample
eligible_qids = [qid for qid, rel in all_qrels.items() if rel["doc_id"] in documents]
example_qid = eligible_qids[0]
example_query_text = all_queries[example_qid]["query"]

output = answer_question(example_query_text, top_k=3, mode="hybrid")

print("question:", output["query"])
print("\ngenerated answer:", output["answer"])
print("reference answer: ", all_answers.get(example_qid, "(none provided)"))

print("\nretrieved chunks used as context:")
for i, r in enumerate(output["retrieved_chunks"], 1):
    title = doc_titles.get(r["doc_id"], r["doc_id"])
    print("\n[{}] {} (section {})".format(i, title, r["section_id"]))
    print(textwrap.fill(r["chunk"][:400], width=90))

## 10. Validation Logs

This dataset ships gold relevance labels (`qrels.json`), which means I can
check retrieval accuracy against actual ground truth instead of just
eyeballing whether the answers look reasonable. For each sample query, I know
exactly which `(doc_id, section_id)` is supposed to be retrieved. That gives a
real **Recall@k**, the fraction of queries where the correct section actually
showed up in the top-k retrieved chunks, plus a side-by-side of the generated
answer against the dataset's reference answer.

In [ ]:
N_VALIDATION_QUERIES = 15
TOP_K = 3

# same filter as before -- can only check queries whose gold doc we actually downloaded
eligible_qids = [qid for qid, rel in all_qrels.items() if rel["doc_id"] in documents]
validation_qids = random.sample(eligible_qids, min(N_VALIDATION_QUERIES, len(eligible_qids)))

validation_log = []
hits = 0

for qid in validation_qids:
    query_text = all_queries[qid]["query"]
    query_type = all_queries[qid].get("type", "unknown")
    gold_doc_id = all_qrels[qid]["doc_id"]
    gold_section_id = all_qrels[qid]["section_id"]
    reference_answer = all_answers.get(qid, "")

    result = answer_question(query_text, top_k=TOP_K, mode="hybrid")
    retrieved_pairs = [(r["doc_id"], r["section_id"]) for r in result["retrieved_chunks"]]

    is_hit = (gold_doc_id, gold_section_id) in retrieved_pairs
    if is_hit:
        hits += 1

    validation_log.append({
        "query": query_text,
        "type": query_type,
        "generated_answer": result["answer"],
        "reference_answer": reference_answer,
        "gold_doc_id": gold_doc_id,
        "gold_section_id": gold_section_id,
        "retrieved_pairs": retrieved_pairs,
        "hit": is_hit,
    })

    tag = "HIT" if is_hit else "MISS"
    print("=" * 90)
    print("[{}] ({}) {}".format(tag, query_type, query_text))
    print("generated:", result["answer"])
    print("reference:", reference_answer)

recall_at_k = hits / len(validation_log)
print("\n" + "=" * 90)
print("Recall@{}: {}/{} = {:.2%}".format(TOP_K, hits, len(validation_log), recall_at_k))

**Validation notes:**
- "Hit" means the exact gold `(doc_id, section_id)` pair showed up somewhere
  in the top-k retrieved chunks for that query. That's a strict, section-level
  check, not just "something relevant-looking came back."
- Misses are worth looking at individually. Usually it's either a chunking
  boundary problem (the answer got split awkwardly) or the sampled hard
  negatives and other papers containing wording that superficially matches the
  query better than the true source does.
- I only downloaded `N_PAPERS` documents, not the full ~1000-paper corpus, so
  this Recall@k reflects retrieval quality within that smaller sample, not the
  full benchmark score. Bump `N_PAPERS` up for a more rigorous number.

## 11. System Metrics Report

A summary of the configuration used across the pipeline: chunking, embeddings,
vector store, retrieval, and the generation model.

In [ ]:
metrics_report = {
    "dataset": {
        "source": "vectara/open_ragbench (Open RAG Benchmark)",
        "papers_downloaded": len(documents),
        "total_sections": sum(len(s) for s in documents.values()),
        "validation_queries_evaluated": len(validation_log),
    },
    "chunking": {
        "chunk_size_chars": 700,
        "chunk_overlap_chars": 100,
        "total_chunks_created": len(all_chunks),
    },
    "embedding": {
        "model": embedding_model_name,
        "embedding_dimension": embedding_dim,
    },
    "vector_store": {
        "tool": "FAISS",
        "index_type": "IndexFlatL2",
        "total_vectors_indexed": index.ntotal,
    },
    "retrieval": {
        "strategies_available": ["vector", "keyword (BM25)", "hybrid"],
        "default_top_k": TOP_K,
        "recall_at_k": "{:.2%}".format(recall_at_k),
    },
    "language_model": {
        "model": "google/flan-t5-base",
        "task": "text2text-generation",
        "max_new_tokens": 150,
    },
}

for section, details in metrics_report.items():
    print("\n" + section.upper())
    for k, v in details.items():
        print(" ", k, "-", v)

## 12. Improvements & Experiments

Things worth trying next, roughly in order of expected payoff:

- **Re-ranking:** run a cross-encoder (e.g. `cross-encoder/ms-marco-MiniLM-L-6-v2`)
  over the top-N hybrid results to reorder them before generation.
- **Chunking strategy:** try sentence-aware or paragraph-aware chunking
  instead of fixed character windows, so chunks stop getting cut mid-sentence.
- **Embedding model:** compare `all-MiniLM-L6-v2` against something bigger,
  like `bge-base-en-v1.5`, and see if the accuracy gain is worth the extra
  compute.
- **Hybrid weighting:** sweep the `alpha` parameter in `hybrid_search` to find
  the best keyword/vector balance for this kind of document.
- **Bigger LLM:** swap `flan-t5-base` for something stronger if GPU or API
  access is available, mainly for more fluent answers.
- **Scale up the sample:** push `N_PAPERS` toward the full corpus (400 gold +
  600 hard negatives) for a more rigorous Recall@k, and include the hard
  negatives specifically to check whether retrieval actually avoids irrelevant
  documents, not just finds the relevant ones.
- **Multimodal retrieval:** the dataset includes base64-encoded images and
  markdown tables per section. Right now I fold tables into text and skip
  images entirely; adding an image embedding model like CLIP would let the
  system handle the text-image and text-table-image queries the benchmark is
  built to test.

## 13. Conclusion

This notebook covers the full RAG pipeline end to end: ingestion, chunking,
embedding, vector storage, retrieval (vector, keyword, and hybrid), and
grounded answer generation, built on real arXiv papers from the Open RAG
Benchmark. Because that dataset comes with gold relevance labels and reference
answers, I could actually validate this against ground truth (Recall@k)
instead of just spot-checking it, and the metrics report above lays out the
full configuration. The ingestion step is modular enough to point at a folder
of your own PDFs, like a resume or notes, instead of the Hub dataset, or to
extend with the re-ranking and multimodal ideas above.